# 05 - Cartographie Interactive - Reseau Electrique du Togo
## CEET Smart Grid - Energy Blackout Prediction
**Objectif :** Visualiser geographiquement les zones a risque avec Folium.

In [11]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap, MarkerCluster, MiniMap
from pathlib import Path
from utils import DATA_PROC, FIGURES

df = pd.read_csv(DATA_PROC / 'ceet_processed.csv')
print(f"Dataset charge : {df.shape}")

Dataset charge : (50000, 75)


### 5.1 Coordonnees des Villes Togolaises

In [12]:
CITY_COORDS = {
    "Lome":    {"lat": 6.1375,  "lon": 1.2123},
    "Kpalime": {"lat": 6.8999,  "lon": 0.6244},
    "Atakpame":{"lat": 7.5833,  "lon": 1.1333},
    "Sokode":  {"lat": 8.9833,  "lon": 1.1333},
    "Kara":    {"lat": 9.5511,  "lon": 1.1861},
    "Dapaong": {"lat": 10.8623, "lon": 0.2061},
    "Tsevie":  {"lat": 6.4271,  "lon": 1.2189},
}

REGION_CENTROIDS = {
    "Maritime": {"lat": 6.30,  "lon": 1.30},
    "Plateaux": {"lat": 7.20,  "lon": 0.90},
    "Centrale": {"lat": 8.65,  "lon": 1.15},
    "Kara":     {"lat": 9.55,  "lon": 1.19},
    "Savanes":  {"lat": 10.50, "lon": 0.30},
    "Lome":     {"lat": 6.14,  "lon": 1.21},
}
print("Coordonnees configurees pour", len(CITY_COORDS), "villes")

Coordonnees configurees pour 7 villes


### 5.2 Agregation des donnees par ville

In [13]:
city_stats = df.groupby('city').agg(
    n_readings=('city','count'),
    blackout_rate=('blackout','mean'),
    avg_risk=('outage_risk','mean'),
    total_shedding=('load_shedding_mw','sum'),
    avg_load=('total_load_mw','mean'),
    overload_rate=('overload','mean'),
    n_blackouts=('blackout','sum'),
).round(3).reset_index()

city_stats['blackout_pct'] = (city_stats['blackout_rate'] * 100).round(2)
city_stats['overload_pct'] = (city_stats['overload_rate'] * 100).round(2)
city_stats['lat'] = city_stats['city'].map(lambda c: CITY_COORDS.get(c, {}).get('lat', 6.14))
city_stats['lon'] = city_stats['city'].map(lambda c: CITY_COORDS.get(c, {}).get('lon', 1.21))

print(city_stats[['city','n_blackouts','blackout_pct','avg_risk','total_shedding']].to_string(index=False))

    city  n_blackouts  blackout_pct  avg_risk  total_shedding
Atakpame          274           3.8    94.704       103640.51
 Dapaong          268           3.7    94.540       103042.10
    Kara          320           4.5    94.642       101474.88
 Kpalime          278           3.9    94.617       103829.87
    Lome          283           4.0    94.429       101424.03
  Sokode          265           3.8    94.657       101000.70
  Tsevie          287           4.0    94.669       104220.37


### 5.3 Carte Principale - Zones a Risque

In [14]:
def risk_color(risk_pct):
    if risk_pct >= 75: return '#E63946'
    if risk_pct >= 50: return '#F4A261'
    if risk_pct >= 30: return '#E9C46A'
    return '#2A9D8F'

def risk_label(risk_pct):
    if risk_pct >= 75: return 'CRITIQUE'
    if risk_pct >= 50: return 'ELEVE'
    if risk_pct >= 30: return 'MODERE'
    return 'FAIBLE'

m = folium.Map(location=[8.0, 1.1], zoom_start=7, tiles='CartoDB dark_matter')

title_html = '''<div style="position:fixed;top:10px;left:50%;transform:translateX(-50%);
z-index:1000;background:rgba(29,53,87,0.92);border-radius:12px;padding:12px 24px;
color:white;font-family:Arial;text-align:center;border:2px solid #E63946;">
<b style="font-size:18px;">CEET Smart Grid - Risque Reseau Electrique Togo</b></div>'''
m.get_root().html.add_child(folium.Element(title_html))

for _, row in city_stats.iterrows():
    color  = risk_color(row['avg_risk'])
    label  = risk_label(row['avg_risk'])
    radius = 10 + row['avg_risk'] / 6

    popup_html = (
        "<div style='font-family:Arial;width:220px;padding:8px;'>"
        f"<h4 style='color:{color};margin:0;'>{row['city']}</h4><hr style='margin:4px 0;'>"
        f"<b>Risque moyen :</b> {row['avg_risk']:.1f}% ({label})<br>"
        f"<b>Taux blackout :</b> {row['blackout_pct']:.2f}%<br>"
        f"<b>Nb blackouts :</b> {int(row['n_blackouts']):,}<br>"
        f"<b>Taux surcharge :</b> {row['overload_pct']:.2f}%<br>"
        f"<b>Delesage total :</b> {row['total_shedding']:.0f} MW<br>"
        f"<b>Charge moy. :</b> {row['avg_load']:.1f} MW"
        "</div>"
    )

    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=radius,
        color=color, fill=True, fill_color=color,
        fill_opacity=0.7, weight=2,
        popup=folium.Popup(popup_html, max_width=240),
        tooltip=f"{row['city']} - Risque: {row['avg_risk']:.1f}%",
    ).add_to(m)

    folium.Marker(
        location=[row['lat'] + 0.08, row['lon']],
        icon=folium.DivIcon(
            html=f'<div style="font-size:10px;color:white;font-weight:bold;text-shadow:1px 1px 2px black;">{row["city"]}</div>',
            icon_size=(80,20),
        )
    ).add_to(m)

legend_html = '''<div style="position:fixed;bottom:30px;right:20px;z-index:1000;
background:rgba(29,53,87,0.92);border-radius:10px;padding:12px;color:white;font-size:12px;">
<b>Niveau de Risque</b><br>
<span style="color:#E63946;">&#9679;</span> Critique (&ge;75%)<br>
<span style="color:#F4A261;">&#9679;</span> Eleve (&ge;50%)<br>
<span style="color:#E9C46A;">&#9679;</span> Modere (&ge;30%)<br>
<span style="color:#2A9D8F;">&#9679;</span> Faible (&lt;30%)</div>'''
m.get_root().html.add_child(folium.Element(legend_html))
MiniMap(toggle_display=True, position='bottomleft').add_to(m)

map_path = Path('../reports/figures/map_risk_zones.html')
m.save(str(map_path))
print(f"Carte sauvegardee : {map_path}")
m

Carte sauvegardee : ..\reports\figures\map_risk_zones.html


### 5.4 Heatmap Geographique - Intensite des Blackouts

In [15]:
m_heat = folium.Map(location=[8.0, 1.1], zoom_start=7, tiles='CartoDB dark_matter')

heat_data = []
for _, row in city_stats.iterrows():
    weight = row['blackout_pct'] / 100
    n_pts  = max(1, int(row['n_blackouts'] // 50))
    for _ in range(n_pts):
        jlat = row['lat'] + np.random.normal(0, 0.15)
        jlon = row['lon'] + np.random.normal(0, 0.15)
        heat_data.append([jlat, jlon, weight])

HeatMap(
    heat_data, min_opacity=0.3, max_opacity=0.9, radius=30, blur=25,
    gradient={0.2:'#2A9D8F', 0.4:'#E9C46A', 0.65:'#F4A261', 1.0:'#E63946'},
).add_to(m_heat)

heat_path = Path('../reports/figures/map_blackout_heatmap.html')
m_heat.save(str(heat_path))
print(f"Heatmap sauvegardee : {heat_path}")
m_heat

Heatmap sauvegardee : ..\reports\figures\map_blackout_heatmap.html


### 5.5 Carte des Surcharges - Cluster de Marqueurs

In [7]:
m_ol = folium.Map(location=[8.0, 1.1], zoom_start=7, tiles='CartoDB positron')
cluster = MarkerCluster(name='Surcharges').add_to(m_ol)

for _, row in city_stats.iterrows():
    n_overloads = int(row['overload_pct'] * row['n_readings'] / 100)
    if n_overloads == 0:
        continue
    ic = 'red' if row['overload_pct'] > 15 else 'orange' if row['overload_pct'] > 8 else 'blue'
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=folium.Popup(
            f"<b>{row['city']}</b><br>Surcharges: {row['overload_pct']:.1f}%<br>N: {n_overloads:,}",
            max_width=180),
        tooltip=f"{row['city']} - Surcharges: {row['overload_pct']:.1f}%",
        icon=folium.Icon(color=ic, icon='bolt', prefix='fa'),
    ).add_to(cluster)

folium.LayerControl().add_to(m_ol)
ol_path = Path('../reports/figures/map_overload_clusters.html')
m_ol.save(str(ol_path))
print(f"Carte surcharges : {ol_path}")
print("\nCartes generees :")
for p in [map_path, heat_path, ol_path]:
    print(f"  {p.name}")
m_ol

Carte surcharges : ..\reports\figures\map_overload_clusters.html

Cartes generees :
  map_risk_zones.html
  map_blackout_heatmap.html
  map_overload_clusters.html
